In [ ]:
import os
import numpy as np
import SimpleITK as sitk
import re

# ==============================
# CONFIGURACIÓN
# ==============================
ct_folder = r"C:\Users\i.redondo\Documents\TFG\TACS"
mask_folder = r"C:\Users\i.redondo\Documents\TFG\SALIDA"

out_ct_folder = r"C:\Users\i.redondo\Documents\TFG\PREPROCESADO_TAC_piel"
out_mask_folder = r"C:\Users\i.redondo\Documents\TFG\PREPROCESADO_MASK_piel"

os.makedirs(out_ct_folder, exist_ok=True)
os.makedirs(out_mask_folder, exist_ok=True)

target_spacing = (2.5, 2.5, 2.5)
target_size = (256, 256, 256)

clip_min = -1000.0
clip_max = 1000.0

# margen extra alrededor de la anatomía segmentada
margin_voxels = 10

# ==============================
# FUNCIONES AUXILIARES
# ==============================
def extract_patient_number(filename):
    m = re.search(r'(\d+)', filename)
    return m.group(1) if m else None

def is_valid_ct_file(filename):
    if not filename.endswith(".nrrd"):
        return False
    low = filename.lower()
    if "canonico" in low or "ortho" in low:
        return False
    return "_pre" in low

def find_mask_for_ct(ct_filename, mask_files):
    patient_num = extract_patient_number(ct_filename)
    if patient_num is None:
        return None

    candidates = []
    for mf in mask_files:
        if not mf.endswith(".nrrd"):
            continue
        if extract_patient_number(mf) == patient_num:
            candidates.append(mf)

    if not candidates:
        return None

    priority = sorted(
        candidates,
        key=lambda x: (
            0 if "labelmap" in x.lower() else 1,
            0 if "etiquetas" in x.lower() else 1,
            len(x)
        )
    )
    return priority[0]

def get_labels(image):
    arr = sitk.GetArrayFromImage(image).astype(np.int16)
    return set(np.unique(arr).tolist())

def almost_equal_tuple(t1, t2, tol=1e-5):
    return all(abs(a - b) < tol for a, b in zip(t1, t2))

# ==============================
# RESAMPLING
# ==============================
def resample_ct(image, out_spacing):
    """
    Remuestrea el TAC a spacing isotrópico.
    Fuera de imagen rellena con clip_min (-1000 HU), para que tras
    normalizar se convierta en 0.0 y se vea negro en Slicer.
    """
    original_spacing = np.array(image.GetSpacing(), dtype=float)
    original_size = np.array(image.GetSize(), dtype=int)

    out_spacing = np.array(out_spacing, dtype=float)
    out_size = np.round(original_size * (original_spacing / out_spacing)).astype(int)

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(tuple(out_spacing.tolist()))
    resampler.SetSize([int(x) for x in out_size])
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetTransform(sitk.Transform())
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetDefaultPixelValue(clip_min)

    return resampler.Execute(image)

def resample_mask_to_reference(mask, reference_image):
    """
    Remuestrea la máscara usando EXACTAMENTE la geometría del TAC remuestreado.
    Esto evita discrepancias en size/spacing/origin/direction.
    """
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(reference_image)
    resampler.SetTransform(sitk.Transform())
    resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    resampler.SetDefaultPixelValue(0)
    return resampler.Execute(mask)

# ==============================
# BOUNDING BOX Y CROP
# ==============================
def compute_bbox_from_mask(mask_image, margin=10):
    """
    Calcula bounding box en coordenadas array (z, y, x)
    para todos los voxeles de máscara > 0.
    """
    arr = sitk.GetArrayFromImage(mask_image)
    coords = np.argwhere(arr > 0)

    if coords.size == 0:
        shape = np.array(arr.shape)
        return np.array([0, 0, 0]), shape

    min_coords = coords.min(axis=0)
    max_coords = coords.max(axis=0) + 1  # exclusivo

    min_coords = np.maximum(min_coords - margin, 0)
    max_coords = np.minimum(max_coords + margin, np.array(arr.shape))

    return min_coords, max_coords

def crop_to_bbox(image, min_coords_zyx, max_coords_zyx):
    """
    Recorta una imagen usando bbox en coords array (z,y,x)
    y corrige el origin.
    """
    arr = sitk.GetArrayFromImage(image)

    cropped = arr[
        min_coords_zyx[0]:max_coords_zyx[0],
        min_coords_zyx[1]:max_coords_zyx[1],
        min_coords_zyx[2]:max_coords_zyx[2]
    ]

    out = sitk.GetImageFromArray(cropped)
    out.SetSpacing(image.GetSpacing())
    out.SetDirection(image.GetDirection())

    start_index_xyz = [
        int(min_coords_zyx[2]),
        int(min_coords_zyx[1]),
        int(min_coords_zyx[0]),
    ]
    new_origin = image.TransformIndexToPhysicalPoint(start_index_xyz)
    out.SetOrigin(new_origin)

    return out

# ==============================
# AJUSTE A 256³
# ==============================
def crop_or_pad_to_target(image, target_size, pad_value):
    """
    Ajusta a tamaño fijo:
    - si sobra, recorta centrado
    - si falta, rellena centrado
    target_size se da como (x,y,z), mientras que el array está en (z,y,x)
    """
    arr = sitk.GetArrayFromImage(image)

    target_zyx = np.array([target_size[2], target_size[1], target_size[0]], dtype=int)
    current_zyx = np.array(arr.shape, dtype=int)

    # RECORTE centrado si el volumen es mayor
    start_crop = np.maximum((current_zyx - target_zyx) // 2, 0)
    end_crop = start_crop + np.minimum(current_zyx, target_zyx)

    arr = arr[
        start_crop[0]:end_crop[0],
        start_crop[1]:end_crop[1],
        start_crop[2]:end_crop[2]
    ]

    # PADDING centrado si el volumen es menor
    current_zyx = np.array(arr.shape, dtype=int)
    pad_before = np.maximum((target_zyx - current_zyx) // 2, 0)
    pad_after = np.maximum(target_zyx - current_zyx - pad_before, 0)

    arr = np.pad(
        arr,
        (
            (pad_before[0], pad_after[0]),
            (pad_before[1], pad_after[1]),
            (pad_before[2], pad_after[2]),
        ),
        mode="constant",
        constant_values=pad_value
    )

    out = sitk.GetImageFromArray(arr)
    out.SetSpacing(image.GetSpacing())
    out.SetDirection(image.GetDirection())

    # corregir origin por el recorte
    start_index_xyz = [
        int(start_crop[2]),
        int(start_crop[1]),
        int(start_crop[0]),
    ]
    new_origin = image.TransformIndexToPhysicalPoint(start_index_xyz)
    out.SetOrigin(new_origin)

    return out

# ==============================
# NORMALIZACIÓN TAC
# ==============================
def normalize_ct(image):
    arr = sitk.GetArrayFromImage(image).astype(np.float32)
    arr = np.clip(arr, clip_min, clip_max)
    arr = (arr - clip_min) / (clip_max - clip_min)
    arr = np.clip(arr, 0.0, 1.0)

    out = sitk.GetImageFromArray(arr)
    out.CopyInformation(image)
    return out

# ==============================
# PROCESO PRINCIPAL
# ==============================
mask_files = [f for f in os.listdir(mask_folder) if f.endswith(".nrrd")]
ct_files = [f for f in os.listdir(ct_folder) if is_valid_ct_file(f)]

print(f"TACs detectados: {len(ct_files)}")
print(f"Máscaras detectadas: {len(mask_files)}")

for ct_filename in sorted(ct_files):
    ct_path = os.path.join(ct_folder, ct_filename)
    mask_filename = find_mask_for_ct(ct_filename, mask_files)

    if mask_filename is None:
        print(f"❌ No encontrada máscara para {ct_filename}")
        continue

    mask_path = os.path.join(mask_folder, mask_filename)
    patient_num = extract_patient_number(ct_filename)

    print(f"\n🔄 Procesando Paciente {patient_num}...")

    ct = sitk.ReadImage(ct_path)
    mask = sitk.ReadImage(mask_path)

    labels_before = get_labels(mask)

    # 1) Remuestrear TAC
    ct_res = resample_ct(ct, target_spacing)

    # 2) Remuestrear máscara usando el TAC como referencia
    mask_res = resample_mask_to_reference(mask, ct_res)

    # 3) Bounding box basada en la máscara remuestreada
    min_coords, max_coords = compute_bbox_from_mask(mask_res, margin=margin_voxels)

    # 4) Crop conjunto
    ct_crop = crop_to_bbox(ct_res, min_coords, max_coords)
    mask_crop = crop_to_bbox(mask_res, min_coords, max_coords)

    # 5) Ajuste a 256x256x256
    # TAC: pad con -1000 HU para que tras normalizar se vea 0.0
    ct_proc = crop_or_pad_to_target(ct_crop, target_size, pad_value=clip_min)

    # Máscara: pad con 0
    mask_proc = crop_or_pad_to_target(mask_crop, target_size, pad_value=0)

    # 6) Normalizar TAC
    ct_proc = normalize_ct(ct_proc)

    # 7) Forzar misma geometría final en la máscara
    mask_proc.CopyInformation(ct_proc)

    # ==========================
    # VERIFICACIONES
    # ==========================
    labels_after = get_labels(mask_proc)
    missing = sorted(labels_before - labels_after)

    if missing:
        print(f"  ⚠ Etiquetas perdidas: {missing}")
    else:
        print("  ✅ Etiquetas preservadas")

    assert ct_proc.GetSize() == mask_proc.GetSize(), "Size TAC y máscara no coincide"
    assert almost_equal_tuple(ct_proc.GetSpacing(), mask_proc.GetSpacing()), "Spacing TAC y máscara no coincide"
    assert almost_equal_tuple(ct_proc.GetDirection(), mask_proc.GetDirection()), "Direction TAC y máscara no coincide"
    assert almost_equal_tuple(ct_proc.GetOrigin(), mask_proc.GetOrigin()), "Origin TAC y máscara no coincide"

    out_ct_name = f"Paciente{patient_num}_pre_proc.nrrd"
    out_mask_name = f"PacienteEtiquetas{patient_num}_labelmap_proc.nrrd"

    sitk.WriteImage(ct_proc, os.path.join(out_ct_folder, out_ct_name))
    sitk.WriteImage(mask_proc, os.path.join(out_mask_folder, out_mask_name))

print("\n🏁 Proceso completado con éxito a 2.5 mm y tamaño 256x256x256.")

import os
import re
import random
import numpy as np
import SimpleITK as sitk

# ==============================
# CONFIGURACIÓN
# ==============================
ct_folder = r"C:\Users\advan\Documents\TFG\PREPROCESADO_CT_piel"
mask_folder = r"C:\Users\advan\Documents\TFG\PREPROCESADO_MASK_piel"

output_root = r"C:\Users\advan\Documents\TFG\DATASET_2p5D_bases_piel"

seed = 42
n_train = 40
n_test = 20

plane = "axial"   # tu Dataset espera axial

# ==============================
# FUNCIONES AUXILIARES
# ==============================
def extract_patient_number(filename):
    m = re.search(r'(\d+)', filename)
    return m.group(1) if m else None

def get_patient_id_from_ct(filename):
    num = extract_patient_number(filename)
    return f"Paciente{num}" if num is not None else None

def find_mask_for_ct(ct_filename, mask_files):
    patient_num = extract_patient_number(ct_filename)
    if patient_num is None:
        return None

    candidates = []
    for mf in mask_files:
        if extract_patient_number(mf) == patient_num:
            candidates.append(mf)

    if not candidates:
        return None

    priority = sorted(
        candidates,
        key=lambda x: (
            0 if "labelmap" in x.lower() else 1,
            len(x)
        )
    )
    return priority[0]

def save_nrrd_slice(array2d, out_path, spacing_xy=(1.0, 1.0), is_mask=False):
    """
    Guarda slice 2D como NRRD.
    """
    img2d = sitk.GetImageFromArray(array2d)

    # En 2D SimpleITK usa spacing (x,y)
    img2d.SetSpacing((float(spacing_xy[0]), float(spacing_xy[1])))

    if is_mask:
        img2d = sitk.Cast(img2d, sitk.sitkInt16)
    else:
        img2d = sitk.Cast(img2d, sitk.sitkFloat32)

    sitk.WriteImage(img2d, out_path)

# ==============================
# DETECTAR PARES TAC-MÁSCARA
# ==============================
ct_files = sorted([f for f in os.listdir(ct_folder) if f.endswith(".nrrd")])
mask_files = sorted([f for f in os.listdir(mask_folder) if f.endswith(".nrrd")])

pairs = []
for ct_filename in ct_files:
    mask_filename = find_mask_for_ct(ct_filename, mask_files)
    if mask_filename is None:
        print(f"❌ No se encontró máscara para {ct_filename}")
        continue

    patient_id = get_patient_id_from_ct(ct_filename)
    if patient_id is None:
        print(f"❌ No se pudo extraer ID de {ct_filename}")
        continue

    pairs.append((
        patient_id,
        os.path.join(ct_folder, ct_filename),
        os.path.join(mask_folder, mask_filename)
    ))

print(f"Pares válidos encontrados: {len(pairs)}")

if len(pairs) < (n_train + n_test):
    raise ValueError(
        f"No hay suficientes pacientes. Encontrados={len(pairs)}, "
        f"necesarios={n_train+n_test}"
    )

# ==============================
# SPLIT TRAIN / TEST
# ==============================
# ==============================
# SPLIT MANUAL TRAIN / TEST
# ==============================

train_patient_ids = [
    "Paciente1",
    "Paciente2",
    "Paciente4",
    "Paciente5",
    "Paciente6",
    "Paciente7",
    "Paciente8",
    "Paciente9",
    "Paciente10",
    "Paciente11",
    "Paciente12",
    "Paciente13",
    "Paciente21",
    "Paciente22",
    "Paciente23",
    "Paciente24",
    "Paciente25",
    "Paciente26",
    "Paciente27",
    "Paciente28",
    "Paciente33",
    "Paciente34",
    "Paciente35",
    "Paciente36",
    "Paciente37",
    "Paciente38",
    "Paciente39",
    "Paciente40",
    "Paciente41",
    "Paciente42",
    "Paciente49",
    "Paciente50",
    "Paciente51",
    "Paciente52",
    "Paciente53",
    "Paciente54",
    "Paciente55",
    "Paciente56",
    "Paciente57",
    "Paciente58",
]

pairs_dict = {patient_id: (patient_id, ct_path, mask_path)
              for patient_id, ct_path, mask_path in pairs}

missing_train = [p for p in train_patient_ids if p not in pairs_dict]
if missing_train:
    raise ValueError(f"Faltan pacientes de TRAIN en los datos disponibles: {missing_train}")

train_pairs = [pairs_dict[p] for p in train_patient_ids]

test_pairs = [
    pairs_dict[p]
    for p in pairs_dict
    if p not in train_patient_ids
]

print(f"Train: {len(train_pairs)} pacientes")
print(f"Test: {len(test_pairs)} pacientes")

# ==============================
# EXPORTAR SLICES
# ==============================
def export_group(pairs_group, split_name):
    for patient_id, ct_path, mask_path in pairs_group:
        print(f"🔄 Exportando {patient_id} -> {split_name}")

        ct = sitk.ReadImage(ct_path)
        mask = sitk.ReadImage(mask_path)

        # Comprobación geométrica básica
        if ct.GetSize() != mask.GetSize():
            print(f"  ❌ Size no coincide en {patient_id}")
            continue

        ct_arr = sitk.GetArrayFromImage(ct)      # [z,y,x]
        mask_arr = sitk.GetArrayFromImage(mask)  # [z,y,x]

        if ct_arr.shape != mask_arr.shape:
            print(f"  ❌ Shape array no coincide en {patient_id}")
            continue

        # spacing en SimpleITK es (x,y,z)
        spacing = ct.GetSpacing()
        spacing_xy = (spacing[0], spacing[1])

        # carpetas destino
        img_patient_dir = os.path.join(output_root, "images", split_name, patient_id, plane)
        mask_patient_dir = os.path.join(output_root, "masks", split_name, patient_id, plane)

        os.makedirs(img_patient_dir, exist_ok=True)
        os.makedirs(mask_patient_dir, exist_ok=True)

        num_slices = ct_arr.shape[0]  # axial => recorremos z

        for z in range(num_slices):
            img_slice = ct_arr[z, :, :].astype(np.float32)
            mask_slice = mask_arr[z, :, :].astype(np.int16)

            slice_name = f"slice_{z:04d}.nrrd"

            img_out_path = os.path.join(img_patient_dir, slice_name)
            mask_out_path = os.path.join(mask_patient_dir, slice_name)

            save_nrrd_slice(img_slice, img_out_path, spacing_xy=spacing_xy, is_mask=False)
            save_nrrd_slice(mask_slice, mask_out_path, spacing_xy=spacing_xy, is_mask=True)

        print(f"  ✅ {num_slices} slices exportados")

export_group(train_pairs, "train")
export_group(test_pairs, "test")

print("\n Dataset 2.5D generado correctamente.")